In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Cross-Region Phase-Amplitude Coupling (PAC) Analysis

Trial-level cross-region PAC for every trial; behavioral metadata (hit/miss, rpe_cat, etc.) is attached for post-hoc condition splits. Apply any trial-level split (e.g. hit vs. miss, rpe_cat) after export.

In [ ]:
"""
Import required libraries for PAC analysis
"""
import numpy as np
import mne
from tensorpac import Pac
from mne_connectivity import seed_target_indices
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import os
from scipy.signal import hilbert

from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df


# Statistical functions for optional cluster tests
from mne.stats import (
    permutation_cluster_test,
    permutation_cluster_1samp_test,
    ttest_ind_no_p,
    ttest_1samp_no_p,
)

warnings.filterwarnings('ignore')


In [ ]:
"""
Add custom analysis utilities to path and import
"""
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import oscillation_utils


In [ ]:
"""
Data loading and subject selection
"""
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

subj_ids = subj_df.MSSMCode.astype(str).values
print(f'Loaded {len(subj_ids)} subjects for analysis')


In [ ]:
"""
Load and preprocess behavior tables via mv_granger_utils
"""
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print(f'learn_df shape: {learn_df.shape}')
print(f'combined_df shape: {combined_df.shape}')
print('combined required columns present:', {'participant', 'condition', 'trials_gamble', 'hits'}.issubset(combined_df.columns))


In [ ]:
"""
Analysis configuration
"""
evs = {
    'gamble_start': [-1.0, 0.0],
    'feedback_start': [0.0, 1.5],
    'baseline_start': [0.0, 0.75],
    'cue_on': [0.0, 1.0],
}
analysis_evs = ['cue_on']
baseline_ev = 'baseline_start'
day = 'Day1'

for ev in analysis_evs:
    if ev not in evs:
        raise ValueError(f'Analysis event "{ev}" is not defined in evs')

print('Analysis events:', analysis_evs)
print('Day:', day)


In [ ]:
"""
Frequency band configuration
"""
phase_frequency = 'theta'
f_pha_start, f_pha_end = 4, 8
f_pha_nb = 15
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4.0, f_pha_mid + f_pha_mid / 4.0]

amp_frequency = 'hfa'
f_amp_start, f_amp_end = 70, 200
f_amp_nb = 20
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8.0, f_amp_mid + f_amp_mid / 8.0]

print(f'Phase band: {phase_frequency} ({f_pha_start}-{f_pha_end} Hz, {f_pha_nb} bins)')
print(f'Amplitude band: {amp_frequency} ({f_amp_start}-{f_amp_end} Hz, {f_amp_nb} bins)')


In [ ]:
"""
Output directory configuration
"""
save_dir_data = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-CrossRegionPAC'
os.makedirs(save_dir_data, exist_ok=True)
os.makedirs(save_dir_plots, exist_ok=True)

print('Data save dir:', save_dir_data)
print('Plot save dir:', save_dir_plots)


In [ ]:
"""
Helper functions for PAC computation, condition-agnostic high-PAC masking, and visualization
"""

def compute_pac_for_pair(pac_obj, ph_sig, amp_sig, sfreq, edges, n_perm=250, random_state=0):
    epoch_pac = pac_obj.filterfit(
        sfreq,
        ph_sig,
        amp_sig,
        n_perm=n_perm,
        n_jobs=-1,
        edges=edges,
        random_state=random_state,
    )

    surr_mean = pac_obj.surrogates.mean(axis=0)
    surr_std = pac_obj.surrogates.std(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        z_scored_pac = (epoch_pac - surr_mean) / surr_std

    return z_scored_pac, epoch_pac


def create_pac_dataframe(
    z_scored_pac,
    subj_id,
    subj_info,
    phase_roi,
    amp_roi,
    event,
    phase_freq,
    amp_freq,
    ph_ch_name,
    amp_ch_name,
    hemi,
    learn_df,
    combined_df,
):
    avg_conn = np.nanmean(z_scored_pac, axis=(0, 1))

    z_scored_pac_sig = z_scored_pac.copy()
    z_scored_pac_sig[np.abs(z_scored_pac_sig) <= 1.96] = np.nan
    sum_conn = np.nansum(z_scored_pac_sig, axis=(0, 1))

    pwise_df = pd.DataFrame({
        'participant': subj_id,
        'age': subj_info['Age'],
        'sex': subj_info['Sex'].strip() if isinstance(subj_info['Sex'], str) else subj_info['Sex'],
        'amp_roi': amp_roi,
        'phase_roi': phase_roi,
        'event': event,
        'band0': phase_freq,
        'band1': amp_freq,
        'trials': np.arange(1, len(sum_conn) + 1),
        'label': f'{ph_ch_name}:{amp_ch_name}',
        'phase_channel': ph_ch_name,
        'amp_channel': amp_ch_name,
        'hemi': hemi,
        'avg_conn': avg_conn,
        'sum_conn': sum_conn,
    })

    if str(event).strip().lower() == 'cue_on':
        cdf = combined_df[combined_df.participant == subj_id].copy()
        if 'condition' in cdf.columns:
            cdf = cdf[cdf['condition'] == 'Day1'].copy()
        if 'trials_mem' in cdf.columns:
            cdf = cdf.rename(columns={'trials_mem': 'trials'})
        pwise_df = pwise_df.merge(cdf, on=['participant', 'trials'], how='left')
    else:
        subj_learn_df = learn_df[learn_df.participant == subj_id].copy()
        pwise_df = pwise_df.merge(subj_learn_df, on=['participant', 'trials'], how='left')

    if 'rpe' in pwise_df.columns:
        denom = 2.0 * np.nanstd(pwise_df['rpe'])
        pwise_df['zrpe'] = (pwise_df['rpe'] - np.nanmean(pwise_df['rpe'])) / denom if np.isfinite(denom) and denom > 0 else np.nan
    if 'DPRIME' in pwise_df.columns:
        denom = 2.0 * np.nanstd(pwise_df['DPRIME'])
        pwise_df['zpm'] = (pwise_df['DPRIME'] - np.nanmean(pwise_df['DPRIME'])) / denom if np.isfinite(denom) and denom > 0 else np.nan

    return pwise_df


def compute_high_pac_mask(trial_pac_z, conditions, high_pac_quantile=0.9, min_high_trials=5):
    """High-PAC trial mask per unique condition (e.g. hit/miss, rpe_cat)."""
    trial_pac_z = np.asarray(trial_pac_z, dtype=float)
    conditions = np.asarray(conditions, dtype=object)
    mask = np.zeros(trial_pac_z.shape[0], dtype=bool)

    for cond in np.unique(conditions):
        if pd.isna(cond) or (isinstance(cond, str) and not str(cond).strip()):
            continue
        cond_mask = (conditions == cond) & np.isfinite(trial_pac_z)
        n_cond = int(cond_mask.sum())
        if n_cond == 0:
            continue

        cond_scores = trial_pac_z[cond_mask]
        threshold = np.nanquantile(cond_scores, high_pac_quantile)
        cond_selected = np.where(cond_mask & (trial_pac_z >= threshold))[0]

        min_take = min(min_high_trials, n_cond)
        if len(cond_selected) < min_take:
            cond_indices = np.where(cond_mask)[0]
            rank_order = np.argsort(cond_scores)[::-1]
            cond_selected = cond_indices[rank_order[:min_take]]

        mask[cond_selected] = True

    return mask


def summarize_by_condition(pwise_df, condition_col='hit_miss', valid_col='alignment_valid'):
    """Per-condition means of avg_conn and sum_conn; optional deltas when exactly two conditions."""
    valid_df = pwise_df[pwise_df[valid_col] == True].copy()
    if condition_col not in valid_df.columns:
        return {}
    cond_vals = valid_df[condition_col].dropna().unique()
    cond_vals = [c for c in cond_vals if (isinstance(c, str) and c.strip()) or (not isinstance(c, str))]
    out = {}
    means_avg = {}
    means_sum = {}
    for cond in cond_vals:
        sub = valid_df[valid_df[condition_col] == cond]
        means_avg[cond] = float(np.nanmean(sub['avg_conn'])) if len(sub) > 0 else np.nan
        means_sum[cond] = float(np.nanmean(sub['sum_conn'])) if len(sub) > 0 else np.nan
        out[f'mean_avg_conn_{cond}'] = means_avg[cond]
        out[f'mean_sum_conn_{cond}'] = means_sum[cond]
    if len(cond_vals) == 2:
        c1, c2 = cond_vals[0], cond_vals[1]
        out[f'delta_{c1}_minus_{c2}_avg_conn'] = (
            means_avg[c1] - means_avg[c2] if np.isfinite(means_avg[c1]) and np.isfinite(means_avg[c2]) else np.nan
        )
        out[f'delta_{c1}_minus_{c2}_sum_conn'] = (
            means_sum[c1] - means_sum[c2] if np.isfinite(means_sum[c1]) and np.isfinite(means_sum[c2]) else np.nan
        )
    return out


def plot_high_pac_lfp_phase_locking(
    phase_sig,
    amp_sig,
    trial_pac_z,
    hit_miss_labels,
    sfreq,
    times,
    f_pha_band,
    f_amp_band,
    out_file,
    title_prefix,
    high_pac_quantile=0.9,
    min_high_trials=5,
):
    phase_sig = np.asarray(phase_sig, dtype=float)
    amp_sig = np.asarray(amp_sig, dtype=float)
    trial_pac_z = np.asarray(trial_pac_z, dtype=float)
    hit_miss_labels = np.asarray(hit_miss_labels, dtype=object)

    if phase_sig.ndim != 2 or amp_sig.ndim != 2:
        return None
    if phase_sig.shape != amp_sig.shape:
        return None
    if phase_sig.shape[0] != len(trial_pac_z):
        return None

    nyquist = sfreq / 2.0
    lf_low = max(float(f_pha_band[0]), 0.5)
    lf_high = min(float(f_pha_band[1]), nyquist - 1.0)
    hf_low = max(float(f_amp_band[0]), 1.0)
    hf_high = min(float(f_amp_band[1]), nyquist - 1.0)

    if lf_high <= lf_low or hf_high <= hf_low:
        return None

    high_mask = compute_high_pac_mask(
        trial_pac_z=trial_pac_z,
        conditions=hit_miss_labels,
        high_pac_quantile=high_pac_quantile,
        min_high_trials=min_high_trials,
    )

    unique_conds = [c for c in np.unique(hit_miss_labels) if pd.notna(c) and (not isinstance(c, str) or c.strip())]
    cond_indices = {
        cond: np.where(high_mask & (hit_miss_labels == cond) & np.isfinite(trial_pac_z))[0]
        for cond in unique_conds
    }

    if all(len(v) == 0 for v in cond_indices.values()):
        return None

    phase_bins = np.linspace(-np.pi, np.pi, 19)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2.0
    n_conds = len(unique_conds)
    fig, axes = plt.subplots(n_conds, 3, figsize=(16, 4 * n_conds), sharex='col')
    if n_conds == 1:
        axes = axes[np.newaxis, :]

    for row_idx, cond in enumerate(unique_conds):
        idx = cond_indices[cond]
        ax_lf, ax_hf, ax_phase = axes[row_idx]

        if len(idx) == 0:
            ax_lf.text(0.5, 0.5, f'No high-PAC {cond} trials', ha='center', va='center')
            ax_hf.text(0.5, 0.5, f'No high-PAC {cond} trials', ha='center', va='center')
            ax_phase.text(0.5, 0.5, f'No high-PAC {cond} trials', ha='center', va='center')
            for ax in (ax_lf, ax_hf, ax_phase):
                ax.set_xticks([])
                ax.set_yticks([])
            continue

        lf_data = mne.filter.filter_data(
            phase_sig[idx],
            sfreq=sfreq,
            l_freq=lf_low,
            h_freq=lf_high,
            verbose=False,
        )
        hf_data = mne.filter.filter_data(
            amp_sig[idx],
            sfreq=sfreq,
            l_freq=hf_low,
            h_freq=hf_high,
            verbose=False,
        )

        hf_env = np.abs(hilbert(hf_data, axis=-1))
        lf_phase = np.angle(hilbert(lf_data, axis=-1))

        mean_lf = np.nanmean(lf_data, axis=0)
        mean_hf_env = np.nanmean(hf_env, axis=0)

        phase_flat = lf_phase.ravel()
        env_flat = hf_env.ravel()
        phase_bin_idx = np.digitize(phase_flat, phase_bins) - 1
        phase_profile = []
        for b in range(len(phase_bins) - 1):
            m = phase_bin_idx == b
            phase_profile.append(np.nanmean(env_flat[m]) if np.any(m) else np.nan)
        phase_profile = np.asarray(phase_profile, dtype=float)

        cond_colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6', '#f39c12']
        clr = cond_colors[row_idx % len(cond_colors)]
        ax_lf.plot(times, mean_lf, color=clr, linewidth=1.5)
        ax_lf.set_title(f'{cond.upper()} LF ({lf_low:.1f}-{lf_high:.1f} Hz) | n={len(idx)}')
        ax_lf.set_ylabel('LF signal')

        ax_hf.plot(times, mean_hf_env, color=clr, linewidth=1.5)
        ax_hf.set_title(f'{cond.upper()} HF envelope ({hf_low:.1f}-{hf_high:.1f} Hz)')
        ax_hf.set_ylabel('HF envelope')

        ax_phase.plot(phase_centers, phase_profile, color=clr, linewidth=1.5)
        ax_phase.set_title(f'{cond.upper()} phase-binned HF envelope')
        ax_phase.set_xlabel('LF phase (rad)')
        ax_phase.set_ylabel('Mean HF envelope')

    axes[-1, 0].set_xlabel('Time (s)')
    axes[-1, 1].set_xlabel('Time (s)')

    fig.suptitle(f'{title_prefix} | High z-scored PAC trials', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(out_file, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return out_file


In [ ]:
"""
Cross-region PAC main loop: trial-level PAC for every trial; behavioral metadata attached for post-hoc splits.
"""
# Analysis parameters
n_perm = 250
buf_ms = 1000
high_pac_quantile = 0.9
min_high_trials = 5

roi_pairs = [
    ('HPC', 'ACC'),
]

pac_obj = Pac(idpac=(2, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

all_dfs = []
subjects_to_process = subj_ids

print('\n' + '=' * 70)
print('Starting CROSS-REGION PAC computation')
print('Subjects:', len(subjects_to_process))
print('Events:', analysis_evs)
print('ROI pairs:', roi_pairs)
print('=' * 70 + '\n')

for subj_idx, subj_id in enumerate(subjects_to_process):
    print(f"[{subj_idx + 1}/{len(subjects_to_process)}] Subject: {subj_id}")
    subj_info = subj_df[subj_df.MSSMCode.astype(str) == str(subj_id)].iloc[0]

    elec_df_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df'
    if not os.path.exists(elec_df_path):
        print('  Missing electrode table:', elec_df_path)
        continue
    elec_df = pd.read_csv(elec_df_path)

    for event in analysis_evs:
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        epochs_file = f'{save_path}/{event}-epo.fif'
        if not os.path.exists(epochs_file):
            print('  Missing epochs file:', epochs_file)
            continue

        print('  Event:', event)
        epochs_reref = mne.read_epochs(epochs_file, preload=True)
        sfreq = float(epochs_reref.info['sfreq'])
        edges = int(buf_ms * (sfreq / 1000.0))

        for phase_roi, amp_roi in roi_pairs:
            print(f'    ROI pair: {phase_roi} -> {amp_roi}')
            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, phase_roi, amp_roi)
            seed_target_df = seed_target_df.rename(columns={'seed': 'phase', 'target': 'amp'})
            pwise_dfs = []

            for hemi in ['l', 'r']:
                if hemi not in seed_target_df.index.tolist():
                    continue

                seed_to_target = seed_target_indices(
                    seed_target_df['phase'][hemi],
                    seed_target_df['amp'][hemi],
                )
                n_pairs = len(seed_to_target[0])
                if n_pairs == 0:
                    continue
                print(f'      {hemi.upper()} hemisphere pairs: {n_pairs}')

                for pair_idx in range(n_pairs):
                    ph_ch_idx = seed_to_target[0][pair_idx]
                    amp_ch_idx = seed_to_target[1][pair_idx]

                    ph_ch_name = epochs_reref.ch_names[ph_ch_idx]
                    amp_ch_name = epochs_reref.ch_names[amp_ch_idx]

                    pac_file = f'{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_{ph_ch_name}-{amp_ch_name}_pac.npy'

                    ph_sig = np.squeeze(epochs_reref.copy().pick(ph_ch_name).get_data())
                    amp_sig = np.squeeze(epochs_reref.copy().pick(amp_ch_name).get_data())

                    z_scored_pac, _ = compute_pac_for_pair(
                        pac_obj=pac_obj,
                        ph_sig=ph_sig,
                        amp_sig=amp_sig,
                        sfreq=sfreq,
                        edges=edges,
                        n_perm=n_perm,
                        random_state=0,
                    )
                    np.save(pac_file, z_scored_pac)

                    trial_pac_z = np.nanmean(z_scored_pac, axis=(0, 1))

                    pwise_df = create_pac_dataframe(
                        z_scored_pac=z_scored_pac,
                        subj_id=str(subj_id),
                        subj_info=subj_info,
                        phase_roi=phase_roi,
                        amp_roi=amp_roi,
                        event=event,
                        phase_freq=phase_frequency,
                        amp_freq=amp_frequency,
                        ph_ch_name=ph_ch_name,
                        amp_ch_name=amp_ch_name,
                        hemi=hemi,
                        learn_df=learn_df,
                        combined_df=combined_df,
                    )

                    pwise_df['trial_pac_z'] = trial_pac_z
                    pwise_dfs.append(pwise_df)

            if len(pwise_dfs) > 0:
                all_df_pt = pd.concat(pwise_dfs, ignore_index=True)
                output_file = f'{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_pac_{event}.csv'
                all_df_pt.to_csv(output_file, index=False)
                all_dfs.append(all_df_pt)
                print('      Saved rows:', len(all_df_pt), '|', output_file)

if len(all_dfs) > 0:
    all_results_df = pd.concat(all_dfs, ignore_index=True)
    combined_output = f'{save_dir_data}/all_subjects_crossregion_pac_combined.csv'
    all_results_df.to_csv(combined_output, index=False)
    print('Saved combined trial-level:', combined_output, '| rows:', len(all_results_df))
else:
    print('No cross-region PAC results to save.')


In [ ]:
z_scored_pac

## Output Checks

In [ ]:
"""
Load and inspect combined trial-level export
"""
trial_file = f'{save_dir_data}/all_subjects_crossregion_pac_combined.csv'

if os.path.exists(trial_file):
    trial_df = pd.read_csv(trial_file)
    print('Trial-level rows:', len(trial_df))
    expected_cols = ['participant', 'trials', 'trial_pac_z', 'hit_miss', 'alignment_valid', 'avg_conn', 'sum_conn']
    print('Required columns present:', {c: (c in trial_df.columns) for c in expected_cols})
    if 'hit_miss' in trial_df.columns:
        print('Unique hit_miss labels:', sorted(trial_df['hit_miss'].dropna().astype(str).unique().tolist()))
    if 'rpe_cat' in trial_df.columns:
        print('Unique rpe_cat labels:', sorted(trial_df['rpe_cat'].dropna().astype(str).unique().tolist()))
else:
    print('Missing trial export:', trial_file)


## Post-hoc trial-level splits

Apply any trial-level split (e.g. `hit_miss`, `rpe_cat`) to the combined trial-level PAC DataFrame. Use `apply_trial_split` for group means and `compute_high_pac_mask` with any condition column for high-PAC subsets.

In [ ]:
def apply_trial_split(trial_df, condition_col, value_col='trial_pac_z', valid_col='alignment_valid'):
    """Group by condition_col and return summary stats (mean, se, n) for value_col. Optionally restrict to valid_col==True."""
    df = trial_df.copy()
    if valid_col in df.columns:
        df = df[df[valid_col] == True]
    if condition_col not in df.columns or value_col not in df.columns:
        return pd.DataFrame()
    g = df.groupby(condition_col, dropna=False)[value_col].agg(['mean', 'sem', 'count'])
    g.columns = [f'{value_col}_mean', f'{value_col}_se', 'n']
    return g.reset_index()

In [ ]:
# Use trial-level DataFrame from memory or from disk (trial_df / all_results_df)
pac_df = trial_df if 'trial_df' in dir() and isinstance(trial_df, pd.DataFrame) and len(trial_df) > 0 else (
    all_results_df if 'all_results_df' in dir() and isinstance(all_results_df, pd.DataFrame) else None
)
if pac_df is None or len(pac_df) == 0:
    print('Run the main loop or load the combined CSV first (trial_df or all_results_df).')
else:
    # Example: summary by hit_miss
    print('Summary by hit_miss (alignment_valid only):')
    display(apply_trial_split(pac_df, 'hit_miss', value_col='trial_pac_z'))
    # Example: summary by rpe_cat if present
    if 'rpe_cat' in pac_df.columns:
        print('Summary by rpe_cat (alignment_valid only):')
        display(apply_trial_split(pac_df, 'rpe_cat', value_col='trial_pac_z'))
    # Per-pair summarize_by_condition (e.g. for one subject/ROI pair)
    example = pac_df.dropna(subset=['hit_miss', 'alignment_valid']).head(80)
    if len(example) > 0:
        s = summarize_by_condition(example, condition_col='hit_miss')
        print('Example summarize_by_condition (hit_miss):', s)

In [ ]:
# Post-hoc high-PAC mask with any condition column (e.g. hit_miss or rpe_cat)
# Example: add is_high_pac for hit_miss split
if pac_df is not None and len(pac_df) > 0 and 'trial_pac_z' in pac_df.columns:
    import seaborn as sns
    # Bar plot by hit_miss
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    valid = pac_df.dropna(subset=['hit_miss', 'trial_pac_z'])
    if len(valid) > 0:
        sns.barplot(data=valid, x='hit_miss', y='trial_pac_z', errorbar='se', ax=ax)
        ax.set_title('trial_pac_z by hit_miss (post-hoc)')
        plt.tight_layout()
        plt.show()
    # Bar plot by rpe_cat if present
    if 'rpe_cat' in pac_df.columns:
        valid_rpe = pac_df.dropna(subset=['rpe_cat', 'trial_pac_z'])
        if len(valid_rpe) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(6, 4))
            sns.barplot(data=valid_rpe, x='rpe_cat', y='trial_pac_z', errorbar='se', ax=ax)
            ax.set_title('trial_pac_z by rpe_cat (post-hoc)')
            plt.tight_layout()
            plt.show()

In [ ]:
sns.barplot(data=all_results_df, x='amp_roi', y='trial_pac_z', hue='hit_miss', errorbar='se')

In [ ]:
import statsmodels.formula.api as smf
smf.mixedlm(formula='hit_miss~ 1 + trial_pac_z', 
data=all_results_df.dropna(subset=['hit_miss', 'trial_pac_z']), groups=['participant'])



In [ ]:
import seaborn as sns
sns.barplot(data=all_results_df, x='hit_miss', y='trial_pac_z', errorbar='se')

## Plot Example High-PAC Pairs

Select top cross-region pairs from the saved trial-level output and plot LF/HF phase-locking examples.

In [ ]:
"""
Plot concrete cross-region examples after a completed PAC run
"""
from IPython.display import display, Image

trial_file = f'{save_dir_data}/all_subjects_crossregion_pac_combined.csv'
if 'trial_df' not in locals():
    if os.path.exists(trial_file):
        trial_df = pd.read_csv(trial_file)
    else:
        raise FileNotFoundError(f'Missing trial export: {trial_file}')

required_cols = [
    'participant', 'event', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel',
    'hemi', 'trials', 'trial_pac_z', 'hit_miss', 'alignment_valid'
]
missing_cols = [c for c in required_cols if c not in trial_df.columns]
if len(missing_cols) > 0:
    raise ValueError(f'Missing required columns in trial_df: {missing_cols}')

N_EXAMPLES = 6
EXAMPLE_DAY = day
EXAMPLE_SCORE = 'mean_trial_pac_z'  # ['mean_trial_pac_z', 'delta_hit_minus_miss_avg_conn']

example_pool = trial_df.copy()
example_pool = example_pool[
    (example_pool['alignment_valid'] == True)
    & (example_pool['hit_miss'].isin(['hit', 'miss']))
    & (np.isfinite(pd.to_numeric(example_pool['trial_pac_z'], errors='coerce')))
].copy()

if len(example_pool) == 0:
    raise RuntimeError('No valid hit/miss trials with finite trial_pac_z found for example plotting.')

group_cols = ['participant', 'event', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel', 'hemi']

pair_score = (
    example_pool
    .groupby(group_cols, as_index=False)
    .agg(
        mean_trial_pac_z=('trial_pac_z', 'mean'),
        n_valid_trials=('trial_pac_z', 'count')
    )
)

delta_df = (
    example_pool
    .groupby(group_cols + ['hit_miss'], as_index=False)['avg_conn']
    .mean()
    .pivot_table(index=group_cols, columns='hit_miss', values='avg_conn', aggfunc='first')
    .reset_index()
)
if 'hit' not in delta_df.columns:
    delta_df['hit'] = np.nan
if 'miss' not in delta_df.columns:
    delta_df['miss'] = np.nan
delta_df['delta_hit_minus_miss_avg_conn'] = delta_df['hit'] - delta_df['miss']

pair_score = pair_score.merge(
    delta_df[group_cols + ['delta_hit_minus_miss_avg_conn']],
    on=group_cols,
    how='left'
)

if EXAMPLE_SCORE not in pair_score.columns:
    raise ValueError(f'Invalid EXAMPLE_SCORE: {EXAMPLE_SCORE}')

pair_score = pair_score.sort_values(EXAMPLE_SCORE, ascending=False).reset_index(drop=True)
example_pairs = pair_score.head(N_EXAMPLES)

example_plot_dir = f'{save_dir_plots}/crossregion_examples'
os.makedirs(example_plot_dir, exist_ok=True)

print('Example pool pairs:', len(pair_score))
print('Plotting top examples by', EXAMPLE_SCORE)
display(example_pairs)

saved_files = []
for _, row in example_pairs.iterrows():
    subj_id = str(row['participant'])
    event = str(row['event'])
    phase_roi = str(row['phase_roi'])
    amp_roi = str(row['amp_roi'])
    ph_ch_name = str(row['phase_channel'])
    amp_ch_name = str(row['amp_channel'])

    epochs_file = (
        f"{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{EXAMPLE_DAY}/{event}-epo.fif"
    )
    if not os.path.exists(epochs_file):
        print('Skipping missing epochs:', epochs_file)
        continue

    epochs_reref = mne.read_epochs(epochs_file, preload=True, verbose=False)
    if (ph_ch_name not in epochs_reref.ch_names) or (amp_ch_name not in epochs_reref.ch_names):
        print('Skipping missing channels:', subj_id, event, ph_ch_name, amp_ch_name)
        continue

    ph_sig = np.squeeze(epochs_reref.copy().pick(ph_ch_name).get_data())
    amp_sig = np.squeeze(epochs_reref.copy().pick(amp_ch_name).get_data())

    pair_df = trial_df[
        (trial_df['participant'].astype(str) == subj_id)
        & (trial_df['event'].astype(str) == event)
        & (trial_df['phase_roi'].astype(str) == phase_roi)
        & (trial_df['amp_roi'].astype(str) == amp_roi)
        & (trial_df['phase_channel'].astype(str) == ph_ch_name)
        & (trial_df['amp_channel'].astype(str) == amp_ch_name)
        & (trial_df['hemi'].astype(str) == str(row['hemi']))
    ].copy()

    n_epochs = len(epochs_reref)
    trial_pac_z_full = np.full(n_epochs, np.nan, dtype=float)
    hit_miss_full = np.array(['unknown'] * n_epochs, dtype=object)

    valid_rows = pair_df.dropna(subset=['trials']).copy()
    if len(valid_rows) == 0:
        print('Skipping pair with no trial rows:', subj_id, event, ph_ch_name, amp_ch_name)
        continue

    trial_idx = pd.to_numeric(valid_rows['trials'], errors='coerce').to_numpy(dtype=float) - 1
    in_range = np.isfinite(trial_idx) & (trial_idx >= 0) & (trial_idx < n_epochs)
    trial_idx = trial_idx[in_range].astype(int)

    trial_vals = pd.to_numeric(valid_rows.loc[in_range, 'trial_pac_z'], errors='coerce').to_numpy(dtype=float)
    trial_lbls = valid_rows.loc[in_range, 'hit_miss'].fillna('unknown').astype(str).to_numpy()

    trial_pac_z_full[trial_idx] = trial_vals
    hit_miss_full[trial_idx] = trial_lbls

    token = '_'.join([
        subj_id, event, phase_roi, amp_roi,
        ph_ch_name.replace('/', '-'), amp_ch_name.replace('/', '-'), str(row['hemi'])
    ])
    out_file = f'{example_plot_dir}/example_highpac_{token}.png'

    saved = plot_high_pac_lfp_phase_locking(
        phase_sig=ph_sig,
        amp_sig=amp_sig,
        trial_pac_z=trial_pac_z_full,
        hit_miss_labels=hit_miss_full,
        sfreq=float(epochs_reref.info['sfreq']),
        times=epochs_reref.times,
        f_pha_band=(f_pha_start, f_pha_end),
        f_amp_band=(f_amp_start, f_amp_end),
        out_file=out_file,
        title_prefix=f'{subj_id} {event} {phase_roi}->{amp_roi} {ph_ch_name}->{amp_ch_name}',
        high_pac_quantile=high_pac_quantile,
        min_high_trials=min_high_trials,
    )

    if saved is not None and os.path.exists(saved):
        saved_files.append(saved)
        print('Saved example:', saved)
        display(Image(filename=saved))

print('Total example plots rendered:', len(saved_files))
